# Taller 1 - EcoMarket + Olist

Chatbot de atención al cliente para EcoMarket, construido sobre el dataset
público de Olist. Este notebook cubre la **Fase 3** del taller (ingeniería
de prompts) y se ejecuta de forma local, usando un modelo de lenguaje
**open-source ejecutado localmente con Ollama** (sin costo ni claves de pago).

**Contenido**
1. Introducción
2. Fuente de datos
3. Carga de datos
4. Exploración de los datos
5. Selección de pedidos
6. Construcción de la base de pruebas
7. Prompt para consulta de pedidos
8. Prompt para devoluciones
9. Configuración del modelo de IA generativa (modelo local con Ollama)
10. Pruebas de los prompts
11. Resultados
12. Conclusiones

## 1. Introducción

EcoMarket es una empresa de e-commerce de productos sostenibles que recibe
miles de consultas diarias de soporte. El 80% son repetitivas (estado de
pedido, devoluciones) y hoy se resuelven con un tiempo de respuesta promedio
de 24 horas.

Este notebook implementa y prueba, con datos reales del dataset de Olist,
los dos prompts propuestos en la Fase 3 del taller:

- **Ejercicio 1:** consulta del estado de un pedido.
- **Ejercicio 2:** evaluación de si un producto puede devolverse.

El modelo de lenguaje utilizado es un modelo **open-source** (por ejemplo,
Llama 3.2) ejecutado localmente con **Ollama**, tal como lo permite la
rúbrica del taller para esta fase práctica.

In [113]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

print("Entorno preparado correctamente")

Entorno preparado correctamente


## 2. Fuente de datos

Usamos el dataset público de [Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
como base de datos de ejemplo para simular los pedidos de EcoMarket.

El notebook se ejecuta **de forma local**: los archivos CSV deben estar
disponibles en disco (no se usa `google.colab`). Ajusta `RUTA_BASE` si
mueves la carpeta del proyecto a otra ubicación.

In [114]:
# Rutas locales del proyecto
RUTA_BASE = Path(
    r"D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI"
    r"\Segundo semestre\IA_Generativa\Taller_1"
)
RUTA_ENTRADA = RUTA_BASE / "Entrada" / "archive"
RUTA_SALIDA = RUTA_BASE / "Salida"

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos de entrada:", RUTA_ENTRADA)
print("Carpeta de resultados:", RUTA_SALIDA)
print("¿Existe la carpeta de entrada?:", RUTA_ENTRADA.exists())

Carpeta de datos de entrada: D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI\Segundo semestre\IA_Generativa\Taller_1\Entrada\archive
Carpeta de resultados: D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI\Segundo semestre\IA_Generativa\Taller_1\Salida
¿Existe la carpeta de entrada?: True


## 3. Carga de datos

In [115]:
archivos = os.listdir(RUTA_ENTRADA)

print("Archivos disponibles en la carpeta de entrada:")
for archivo in archivos:
    if archivo.endswith(".csv"):
        print(archivo)

Archivos disponibles en la carpeta de entrada:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [116]:
archivos_csv = sorted([
    archivo for archivo in os.listdir(RUTA_ENTRADA)
    if archivo.endswith(".csv")
])

for archivo in archivos_csv:
    df = pd.read_csv(RUTA_ENTRADA / archivo)

    print("=" * 80)
    print(f"ARCHIVO: {archivo}")
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")
    print("Variables:")
    print(list(df.columns))
    print()

ARCHIVO: olist_customers_dataset.csv
Filas: 99,441
Columnas: 5
Variables:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ARCHIVO: olist_geolocation_dataset.csv
Filas: 1,000,163
Columnas: 5
Variables:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ARCHIVO: olist_order_items_dataset.csv
Filas: 112,650
Columnas: 7
Variables:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ARCHIVO: olist_order_payments_dataset.csv
Filas: 103,886
Columnas: 5
Variables:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

ARCHIVO: olist_order_reviews_dataset.csv
Filas: 99,224
Columnas: 7
Variables:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ARCHIVO: olist_orders_dataset.csv
Filas: 99,441
Column

## 4. Exploración de los datos

In [117]:
# Cargar la tabla principal de pedidos

pedidos = pd.read_csv(RUTA_ENTRADA / "olist_orders_dataset.csv")

print("Tabla de pedidos cargada correctamente")
print(f"Registros: {len(pedidos):,}")
print(f"Columnas: {len(pedidos.columns)}")

pedidos.head()

Tabla de pedidos cargada correctamente
Registros: 99,441
Columnas: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [118]:
# Estados reales de los pedidos
estados = pedidos["order_status"].value_counts()
# Traducción de los estados para visualización
traduccion_estados = {
    "delivered": "Entregado",
    "shipped": "Enviado",
    "canceled": "Cancelado",
    "unavailable": "No disponible",
    "invoiced": "Facturado",
    "processing": "En procesamiento",
    "created": "Creado",
    "approved": "Aprobado"
}
estados_espanol = estados.rename(index=traduccion_estados)
print("Estados de pedidos en español:")
print(estados_espanol)

Estados de pedidos en español:
order_status
Entregado           96478
Enviado              1107
Cancelado             625
No disponible         609
Facturado             314
En procesamiento      301
Creado                  5
Aprobado                2
Name: count, dtype: int64


## 5. Selección de pedidos

In [119]:
# Seleccionar 10 pedidos reales para las pruebas

cantidades = {
    "delivered": 3,     # Entregado
    "shipped": 2,       # Enviado
    "canceled": 2,      # Cancelado
    "unavailable": 1,   # No disponible
    "invoiced": 1,      # Facturado
    "processing": 1     # En procesamiento
}

seleccionados = []

for estado, cantidad in cantidades.items():
    muestra = pedidos[pedidos["order_status"] == estado].head(cantidad)
    seleccionados.append(muestra)

pedidos_prueba = pd.concat(seleccionados, ignore_index=True)

print("Total de pedidos seleccionados:", len(pedidos_prueba))

Total de pedidos seleccionados: 10


In [120]:
# Crear una tabla de presentación en español

pedidos_presentacion = pedidos_prueba[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

# Traducir estados
pedidos_presentacion["estado_espanol"] = (
    pedidos_presentacion["order_status"]
    .map(traduccion_estados)
)

# Renombrar columnas
pedidos_presentacion = pedidos_presentacion.rename(columns={
    "order_id": "Numero_pedido",
    "order_status": "Estado_original",
    "order_purchase_timestamp": "Fecha_compra",
    "order_delivered_carrier_date": "Fecha_entrega_transportadora",
    "order_delivered_customer_date": "Fecha_entrega_cliente",
    "order_estimated_delivery_date": "Fecha_entrega_estimada",
    "estado_espanol": "Estado"
})

# Ordenar columnas
pedidos_presentacion = pedidos_presentacion[[
    "Numero_pedido",
    "Estado",
    "Estado_original",
    "Fecha_compra",
    "Fecha_entrega_transportadora",
    "Fecha_entrega_cliente",
    "Fecha_entrega_estimada"
]]

pedidos_presentacion

,Numero_pedido,Estado,Estado_original,Fecha_compra,Fecha_entrega_transportadora,Fecha_entrega_cliente,Fecha_entrega_estimada
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,delivered,2017-10-02 10:56:33,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,delivered,2018-07-24 20:41:37,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,delivered,2018-08-08 08:38:49,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,shipped,2018-06-04 16:44:48,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
4,6942b8da583c2f9957e990d028607019,Enviado,shipped,2018-01-10 11:33:07,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,canceled,2018-08-04 14:29:27,NaN,NaN,2018-08-14 00:00:00
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,canceled,2018-01-26 21:34:08,2018-01-29 22:33:25,NaN,2018-02-22 00:00:00
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,unavailable,2017-11-16 15:09:28,NaN,NaN,2017-12-05 00:00:00
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,invoiced,2017-04-11 12:22:08,NaN,NaN,2017-05-09 00:00:00
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,processing,2017-09-03 14:22:03,NaN,NaN,2017-10-03 00:00:00


## 6. Construcción de la base de pruebas

In [121]:
# Cargar información de artículos y productos

items = pd.read_csv(RUTA_ENTRADA / "olist_order_items_dataset.csv")
productos = pd.read_csv(RUTA_ENTRADA / "olist_products_dataset.csv")

print("Tabla de artículos:", items.shape)
print("Tabla de productos:", productos.shape)

Tabla de artículos: (112650, 7)
Tabla de productos: (32951, 9)


In [122]:
# Obtener los artículos correspondientes a nuestros 10 pedidos

items_prueba = items[
    items["order_id"].isin(pedidos_prueba["order_id"])
].copy()

print("Artículos encontrados para nuestros 10 pedidos:", len(items_prueba))

items_prueba

Artículos encontrados para nuestros 10 pedidos: 9


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
8509,136cce7faa42fdb2cefd53fdc79a6098,1,a1804276d9941ac0733cfd409f5206eb,dc8798cbf453b7e0f98745e396cc5616,2017-04-19 13:25:17,49.90,16.05
9501,15bed8e2fec7fdbadb186b57c46c92f2,1,61d52f4882421048afd530db53d6f230,fa74b2f3287d296e9fbd2cc80f2d1cf1,2017-09-20 14:30:09,125.90,12.38
12171,1b9ecfe83cdc259250e1a8aca174f0ad,1,ad673c1cd02b966e931f9db4fdc34791,9646c3513289980f17226a2fc4720dbd,2018-08-14 04:10:26,25.00,8.34
31504,47770eb9100c2d0c44946d9cf07ec65d,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
36896,53cdb2fc8bc7dce0b6741e2150273451,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
46325,6942b8da583c2f9957e990d028607019,1,ee0c1cf2fbeae95205b4aa506f1469f0,cc419e0650a3c5ba77189a1882b7556a,2018-01-18 02:32:30,53.99,15.13
49872,714fb133a6730ab81fa1d3c1b2007291,1,a0b7d5a992ccda646f2d34e418fff5a0,95f83f51203c626648c875dd41874c7f,2018-02-01 21:58:39,69.90,26.11
100785,e481f51cbdc54678b7cc49136f2d6af7,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
104935,ee64d42b8cf066f35eac1cf57de1aa85,1,c50ca07e9e4db9ea5011f06802c0aea0,e9779976487b77c6d4ac45f75ec7afe9,2018-06-13 04:30:33,14.49,7.87


In [123]:
# Relacionar los artículos con la información de los productos

items_productos_prueba = items_prueba.merge(
    productos,
    on="product_id",
    how="left"
)

print("Registros después de relacionar artículos y productos:",
      len(items_productos_prueba))

items_productos_prueba[[
    "order_id",
    "product_id",
    "product_category_name",
    "price",
    "freight_value"
]]

Registros después de relacionar artículos y productos: 9


,order_id,product_id,product_category_name,price,freight_value
0,136cce7faa42fdb2cefd53fdc79a6098,a1804276d9941ac0733cfd409f5206eb,NaN,49.90,16.05
1,15bed8e2fec7fdbadb186b57c46c92f2,61d52f4882421048afd530db53d6f230,perfumaria,125.90,12.38
2,1b9ecfe83cdc259250e1a8aca174f0ad,ad673c1cd02b966e931f9db4fdc34791,informatica_acessorios,25.00,8.34
3,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,automotivo,159.90,19.22
4,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,perfumaria,118.70,22.76
5,6942b8da583c2f9957e990d028607019,ee0c1cf2fbeae95205b4aa506f1469f0,perfumaria,53.99,15.13
6,714fb133a6730ab81fa1d3c1b2007291,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,69.90,26.11
7,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,utilidades_domesticas,29.99,8.72
8,ee64d42b8cf066f35eac1cf57de1aa85,c50ca07e9e4db9ea5011f06802c0aea0,beleza_saude,14.49,7.87


In [124]:
# Cargar la tabla de traducción de categorías

traducciones = pd.read_csv(RUTA_ENTRADA / "product_category_name_translation.csv")

print("Registros de traducción:", len(traducciones))
print("Columnas:", list(traducciones.columns))

traducciones.head(10)

Registros de traducción: 71
Columnas: ['product_category_name', 'product_category_name_english']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [125]:
# Unir las categorías en inglés de Olist

items_productos_prueba = items_productos_prueba.merge(
    traducciones,
    on="product_category_name",
    how="left"
)

print("Información de productos con traducción:")
print(
    items_productos_prueba[
        [
            "order_id",
            "product_id",
            "product_category_name",
            "product_category_name_english",
            "price",
            "freight_value"
        ]
    ].to_string(index=False)
)

Información de productos con traducción:
                        order_id                       product_id  product_category_name product_category_name_english  price  freight_value
136cce7faa42fdb2cefd53fdc79a6098 a1804276d9941ac0733cfd409f5206eb                    NaN                           NaN  49.90          16.05
15bed8e2fec7fdbadb186b57c46c92f2 61d52f4882421048afd530db53d6f230             perfumaria                     perfumery 125.90          12.38
1b9ecfe83cdc259250e1a8aca174f0ad ad673c1cd02b966e931f9db4fdc34791 informatica_acessorios         computers_accessories  25.00           8.34
47770eb9100c2d0c44946d9cf07ec65d aa4383b373c6aca5d8797843e5594415             automotivo                          auto 159.90          19.22
53cdb2fc8bc7dce0b6741e2150273451 595fac2a385ac33a80bd5114aec74eb8             perfumaria                     perfumery 118.70          22.76
6942b8da583c2f9957e990d028607019 ee0c1cf2fbeae95205b4aa506f1469f0             perfumaria                     perf

In [126]:
# Traducción de categorías al español para la presentación del taller

categorias_espanol = {
    "perfumaria": "Perfumería",
    "informatica_acessorios": "Informática y accesorios",
    "automotivo": "Automotriz",
    "moveis_decoracao": "Muebles y decoración",
    "utilidades_domesticas": "Utilidades domésticas",
    "beleza_saude": "Belleza y salud"
}

items_productos_prueba["categoria_espanol"] = (
    items_productos_prueba["product_category_name"]
    .map(categorias_espanol)
    .fillna("Sin categoría registrada")
)

items_productos_prueba[[
    "order_id",
    "product_category_name",
    "product_category_name_english",
    "categoria_espanol",
    "price",
    "freight_value"
]]

,order_id,product_category_name,product_category_name_english,categoria_espanol,price,freight_value
0,136cce7faa42fdb2cefd53fdc79a6098,NaN,NaN,Sin categoría registrada,49.90,16.05
1,15bed8e2fec7fdbadb186b57c46c92f2,perfumaria,perfumery,Perfumería,125.90,12.38
2,1b9ecfe83cdc259250e1a8aca174f0ad,informatica_acessorios,computers_accessories,Informática y accesorios,25.00,8.34
3,47770eb9100c2d0c44946d9cf07ec65d,automotivo,auto,Automotriz,159.90,19.22
4,53cdb2fc8bc7dce0b6741e2150273451,perfumaria,perfumery,Perfumería,118.70,22.76
5,6942b8da583c2f9957e990d028607019,perfumaria,perfumery,Perfumería,53.99,15.13
6,714fb133a6730ab81fa1d3c1b2007291,moveis_decoracao,furniture_decor,Muebles y decoración,69.90,26.11
7,e481f51cbdc54678b7cc49136f2d6af7,utilidades_domesticas,housewares,Utilidades domésticas,29.99,8.72
8,ee64d42b8cf066f35eac1cf57de1aa85,beleza_saude,health_beauty,Belleza y salud,14.49,7.87


In [127]:
# Unir los pedidos con la información de sus productos

pedidos_completos = pedidos_prueba.merge(
    items_productos_prueba,
    on="order_id",
    how="left"
)

print("Registros de la tabla final:", len(pedidos_completos))
print("Columnas disponibles:")
print(list(pedidos_completos.columns))

Registros de la tabla final: 10
Columnas disponibles:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'categoria_espanol']


In [128]:
# Crear la base de datos limpia para las pruebas del chatbot

base_chatbot = pedidos_completos[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "categoria_espanol",
    "price",
    "freight_value"
]].copy()

# Traducir el estado al español
base_chatbot["estado_espanol"] = (
    base_chatbot["order_status"]
    .map(traduccion_estados)
)

# Renombrar las columnas para facilitar su uso
base_chatbot = base_chatbot.rename(columns={
    "order_id": "numero_pedido",
    "order_status": "estado_original",
    "order_purchase_timestamp": "fecha_compra",
    "order_estimated_delivery_date": "fecha_entrega_estimada",
    "order_delivered_customer_date": "fecha_entrega_real",
    "categoria_espanol": "categoria_producto",
    "price": "precio_producto",
    "freight_value": "costo_transporte"
})

# Organizar columnas
base_chatbot = base_chatbot[[
    "numero_pedido",
    "estado_espanol",
    "estado_original",
    "fecha_compra",
    "fecha_entrega_estimada",
    "fecha_entrega_real",
    "categoria_producto",
    "precio_producto",
    "costo_transporte"
]]

base_chatbot

,numero_pedido,estado_espanol,estado_original,fecha_compra,fecha_entrega_estimada,fecha_entrega_real,categoria_producto,precio_producto,costo_transporte
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,delivered,2017-10-02 10:56:33,2017-10-18 00:00:00,2017-10-10 21:25:13,Utilidades domésticas,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,delivered,2018-07-24 20:41:37,2018-08-13 00:00:00,2018-08-07 15:27:45,Perfumería,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,delivered,2018-08-08 08:38:49,2018-09-04 00:00:00,2018-08-17 18:06:29,Automotriz,159.90,19.22
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,shipped,2018-06-04 16:44:48,2018-06-28 00:00:00,NaN,Belleza y salud,14.49,7.87
4,6942b8da583c2f9957e990d028607019,Enviado,shipped,2018-01-10 11:33:07,2018-02-07 00:00:00,NaN,Perfumería,53.99,15.13
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,canceled,2018-08-04 14:29:27,2018-08-14 00:00:00,NaN,Informática y accesorios,25.00,8.34
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,canceled,2018-01-26 21:34:08,2018-02-22 00:00:00,NaN,Muebles y decoración,69.90,26.11
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,unavailable,2017-11-16 15:09:28,2017-12-05 00:00:00,NaN,NaN,NaN,NaN
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,invoiced,2017-04-11 12:22:08,2017-05-09 00:00:00,NaN,Sin categoría registrada,49.90,16.05
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,processing,2017-09-03 14:22:03,2017-10-03 00:00:00,NaN,Perfumería,125.90,12.38


In [129]:
# Verificación de los 10 registros

print("Número total de pedidos:", len(base_chatbot))

print("\nTipos de datos:")
print(base_chatbot.dtypes)

print("\nTabla completa:")
display(base_chatbot)

Número total de pedidos: 10

Tipos de datos:
numero_pedido              object
estado_espanol             object
estado_original            object
fecha_compra               object
fecha_entrega_estimada     object
fecha_entrega_real         object
categoria_producto         object
precio_producto           float64
costo_transporte          float64
dtype: object

Tabla completa:


,numero_pedido,estado_espanol,estado_original,fecha_compra,fecha_entrega_estimada,fecha_entrega_real,categoria_producto,precio_producto,costo_transporte
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,delivered,2017-10-02 10:56:33,2017-10-18 00:00:00,2017-10-10 21:25:13,Utilidades domésticas,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,delivered,2018-07-24 20:41:37,2018-08-13 00:00:00,2018-08-07 15:27:45,Perfumería,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,delivered,2018-08-08 08:38:49,2018-09-04 00:00:00,2018-08-17 18:06:29,Automotriz,159.90,19.22
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,shipped,2018-06-04 16:44:48,2018-06-28 00:00:00,NaN,Belleza y salud,14.49,7.87
4,6942b8da583c2f9957e990d028607019,Enviado,shipped,2018-01-10 11:33:07,2018-02-07 00:00:00,NaN,Perfumería,53.99,15.13
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,canceled,2018-08-04 14:29:27,2018-08-14 00:00:00,NaN,Informática y accesorios,25.00,8.34
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,canceled,2018-01-26 21:34:08,2018-02-22 00:00:00,NaN,Muebles y decoración,69.90,26.11
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,unavailable,2017-11-16 15:09:28,2017-12-05 00:00:00,NaN,NaN,NaN,NaN
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,invoiced,2017-04-11 12:22:08,2017-05-09 00:00:00,NaN,Sin categoría registrada,49.90,16.05
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,processing,2017-09-03 14:22:03,2017-10-03 00:00:00,NaN,Perfumería,125.90,12.38


In [130]:
# Guardar la base de pruebas en Excel

archivo_excel = RUTA_SALIDA / "Taller_1_EcoMarket_Datos.xlsx"

with pd.ExcelWriter(archivo_excel, engine="openpyxl") as writer:
    base_chatbot.to_excel(
        writer,
        sheet_name="Pedidos_Prueba",
        index=False
    )

print(f"Archivo creado correctamente: {archivo_excel}")

Archivo creado correctamente: D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI\Segundo semestre\IA_Generativa\Taller_1\Salida\Taller_1_EcoMarket_Datos.xlsx


In [131]:
# Comprobar el archivo Excel generado

archivo_excel = RUTA_SALIDA / "Taller_1_EcoMarket_Datos.xlsx"

print("¿El archivo existe?:", archivo_excel.exists())

# Leer nuevamente el Excel
verificacion = pd.read_excel(archivo_excel, sheet_name="Pedidos_Prueba")

print("\nNúmero de registros:", len(verificacion))
print("Número de columnas:", len(verificacion.columns))

print("\nContenido del Excel:")
display(verificacion)

¿El archivo existe?: True

Número de registros: 10
Número de columnas: 9

Contenido del Excel:


,numero_pedido,estado_espanol,estado_original,fecha_compra,fecha_entrega_estimada,fecha_entrega_real,categoria_producto,precio_producto,costo_transporte
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,delivered,2017-10-02 10:56:33,2017-10-18 00:00:00,2017-10-10 21:25:13,Utilidades domésticas,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,delivered,2018-07-24 20:41:37,2018-08-13 00:00:00,2018-08-07 15:27:45,Perfumería,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,delivered,2018-08-08 08:38:49,2018-09-04 00:00:00,2018-08-17 18:06:29,Automotriz,159.90,19.22
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,shipped,2018-06-04 16:44:48,2018-06-28 00:00:00,NaN,Belleza y salud,14.49,7.87
4,6942b8da583c2f9957e990d028607019,Enviado,shipped,2018-01-10 11:33:07,2018-02-07 00:00:00,NaN,Perfumería,53.99,15.13
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,canceled,2018-08-04 14:29:27,2018-08-14 00:00:00,NaN,Informática y accesorios,25.00,8.34
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,canceled,2018-01-26 21:34:08,2018-02-22 00:00:00,NaN,Muebles y decoración,69.90,26.11
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,unavailable,2017-11-16 15:09:28,2017-12-05 00:00:00,NaN,NaN,NaN,NaN
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,invoiced,2017-04-11 12:22:08,2017-05-09 00:00:00,NaN,Sin categoría registrada,49.90,16.05
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,processing,2017-09-03 14:22:03,2017-10-03 00:00:00,NaN,Perfumería,125.90,12.38


In [132]:
# Preparar la base de prueba para el chatbot

base_prueba = base_chatbot.copy()

# Convertir fechas a formato datetime
base_prueba["fecha_compra"] = pd.to_datetime(
    base_prueba["fecha_compra"],
    errors="coerce"
)

base_prueba["fecha_entrega_estimada"] = pd.to_datetime(
    base_prueba["fecha_entrega_estimada"],
    errors="coerce"
)

base_prueba["fecha_entrega_real"] = pd.to_datetime(
    base_prueba["fecha_entrega_real"],
    errors="coerce"
)

# Crear una respuesta esperada según el estado real
respuestas_estado = {
    "Entregado": "El pedido fue entregado.",
    "Enviado": "El pedido fue enviado y aún no registra entrega al cliente.",
    "Cancelado": "El pedido fue cancelado.",
    "No disponible": "El pedido aparece como no disponible en el registro.",
    "Facturado": "El pedido fue facturado y aún no registra entrega al cliente.",
    "En procesamiento": "El pedido se encuentra en procesamiento y aún no registra entrega al cliente."
}

base_prueba["respuesta_esperada"] = (
    base_prueba["estado_espanol"].map(respuestas_estado)
)

print("Base de prueba preparada correctamente.")
print("Registros:", len(base_prueba))
print("Columnas:", len(base_prueba.columns))

display(base_prueba)

Base de prueba preparada correctamente.
Registros: 10
Columnas: 10


,numero_pedido,estado_espanol,estado_original,fecha_compra,fecha_entrega_estimada,fecha_entrega_real,categoria_producto,precio_producto,costo_transporte,respuesta_esperada
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,delivered,2017-10-02 10:56:33,2017-10-18,2017-10-10 21:25:13,Utilidades domésticas,29.99,8.72,El pedido fue entregado.
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,delivered,2018-07-24 20:41:37,2018-08-13,2018-08-07 15:27:45,Perfumería,118.70,22.76,El pedido fue entregado.
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,delivered,2018-08-08 08:38:49,2018-09-04,2018-08-17 18:06:29,Automotriz,159.90,19.22,El pedido fue entregado.
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,shipped,2018-06-04 16:44:48,2018-06-28,NaT,Belleza y salud,14.49,7.87,El pedido fue enviado y aún no registra entreg...
4,6942b8da583c2f9957e990d028607019,Enviado,shipped,2018-01-10 11:33:07,2018-02-07,NaT,Perfumería,53.99,15.13,El pedido fue enviado y aún no registra entreg...
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,canceled,2018-08-04 14:29:27,2018-08-14,NaT,Informática y accesorios,25.00,8.34,El pedido fue cancelado.
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,canceled,2018-01-26 21:34:08,2018-02-22,NaT,Muebles y decoración,69.90,26.11,El pedido fue cancelado.
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,unavailable,2017-11-16 15:09:28,2017-12-05,NaT,NaN,NaN,NaN,El pedido aparece como no disponible en el reg...
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,invoiced,2017-04-11 12:22:08,2017-05-09,NaT,Sin categoría registrada,49.90,16.05,El pedido fue facturado y aún no registra entr...
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,processing,2017-09-03 14:22:03,2017-10-03,NaT,Perfumería,125.90,12.38,El pedido se encuentra en procesamiento y aún ...


In [133]:
# Función básica para consultar el estado de un pedido

def consultar_pedido(numero_pedido):
    resultado = base_prueba[
        base_prueba["numero_pedido"] == numero_pedido
    ]

    if resultado.empty:
        return "No se encontró el pedido en la base de datos."

    pedido = resultado.iloc[0]

    return (
        f"Pedido: {pedido['numero_pedido']}\n"
        f"Estado: {pedido['estado_espanol']}\n"
        f"Fecha de compra: {pedido['fecha_compra']}\n"
        f"Fecha estimada de entrega: {pedido['fecha_entrega_estimada']}"
    )


# Probar con un pedido REAL de nuestra base
numero_prueba = "e481f51cbdc54678b7cc49136f2d6af7"

respuesta = consultar_pedido(numero_prueba)

print(respuesta)

Pedido: e481f51cbdc54678b7cc49136f2d6af7
Estado: Entregado
Fecha de compra: 2017-10-02 10:56:33
Fecha estimada de entrega: 2017-10-18 00:00:00


## 7. Prompt para consulta de pedidos (Ejercicio 1 - Fase 3)

In [134]:
# Prompt base para el chatbot de EcoMarket
# Fase 3 - Ejercicio 1: Consulta de estado de pedido

prompt_estado_pedido = """
Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es informar al cliente sobre el estado de su pedido
utilizando exclusivamente la información proporcionada en la
base de datos.

REGLAS:
1. No inventes información.
2. No modifiques el estado registrado del pedido.
3. Si el pedido no aparece en la base de datos, indícalo claramente.
4. Responde en español.
5. Utiliza un tono amable, claro y profesional.
6. Si existe una fecha estimada de entrega, comunícala.
7. Si no existe una fecha registrada, indica que no está disponible.
8. No proporciones información que no esté presente en los datos.

DATOS DEL PEDIDO:
{datos_pedido}

PREGUNTA DEL CLIENTE:
{pregunta_cliente}

Genera una respuesta breve y útil para el cliente.
"""

# Ejemplo utilizando nuestro pedido real
pedido = base_prueba[
    base_prueba["numero_pedido"] == "e481f51cbdc54678b7cc49136f2d6af7"
].iloc[0]

datos_pedido = pedido.to_dict()

pregunta_cliente = "¿Cuál es el estado de mi pedido?"

prompt_generado = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

print(prompt_generado)


Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es informar al cliente sobre el estado de su pedido
utilizando exclusivamente la información proporcionada en la
base de datos.

REGLAS:
1. No inventes información.
2. No modifiques el estado registrado del pedido.
3. Si el pedido no aparece en la base de datos, indícalo claramente.
4. Responde en español.
5. Utiliza un tono amable, claro y profesional.
6. Si existe una fecha estimada de entrega, comunícala.
7. Si no existe una fecha registrada, indica que no está disponible.
8. No proporciones información que no esté presente en los datos.

DATOS DEL PEDIDO:
{'numero_pedido': 'e481f51cbdc54678b7cc49136f2d6af7', 'estado_espanol': 'Entregado', 'estado_original': 'delivered', 'fecha_compra': Timestamp('2017-10-02 10:56:33'), 'fecha_entrega_estimada': Timestamp('2017-10-18 00:00:00'), 'fecha_entrega_real': Timestamp('2017-10-10 21:25:13'), 'categoria_producto': 'Utilidades domésticas', 'precio_producto': 29.99, 'c

In [135]:
# Respuesta esperada del asistente para el pedido real

respuesta_asistente = (
    "Hola. Tu pedido e481f51cbdc54678b7cc49136f2d6af7 "
    "se encuentra en estado Entregado. "
    "La fecha de entrega registrada fue el 10 de octubre de 2017."
)

print("Respuesta del asistente:")
print(respuesta_asistente)

Respuesta del asistente:
Hola. Tu pedido e481f51cbdc54678b7cc49136f2d6af7 se encuentra en estado Entregado. La fecha de entrega registrada fue el 10 de octubre de 2017.


In [136]:
# Probar el chatbot con los 10 pedidos reales (respuesta esperada, sin LLM todavía)

for _, pedido in base_prueba.iterrows():

    print("=" * 80)
    print(f"PEDIDO: {pedido['numero_pedido']}")
    print(f"ESTADO REAL: {pedido['estado_espanol']}")

    if pd.notna(pedido["fecha_entrega_real"]):
        fecha_real = pedido["fecha_entrega_real"].strftime("%d/%m/%Y")
        print(f"FECHA DE ENTREGA: {fecha_real}")
    else:
        print("FECHA DE ENTREGA: No registrada")

    print(f"RESPUESTA ESPERADA: {pedido['respuesta_esperada']}")

PEDIDO: e481f51cbdc54678b7cc49136f2d6af7
ESTADO REAL: Entregado
FECHA DE ENTREGA: 10/10/2017
RESPUESTA ESPERADA: El pedido fue entregado.
PEDIDO: 53cdb2fc8bc7dce0b6741e2150273451
ESTADO REAL: Entregado
FECHA DE ENTREGA: 07/08/2018
RESPUESTA ESPERADA: El pedido fue entregado.
PEDIDO: 47770eb9100c2d0c44946d9cf07ec65d
ESTADO REAL: Entregado
FECHA DE ENTREGA: 17/08/2018
RESPUESTA ESPERADA: El pedido fue entregado.
PEDIDO: ee64d42b8cf066f35eac1cf57de1aa85
ESTADO REAL: Enviado
FECHA DE ENTREGA: No registrada
RESPUESTA ESPERADA: El pedido fue enviado y aún no registra entrega al cliente.
PEDIDO: 6942b8da583c2f9957e990d028607019
ESTADO REAL: Enviado
FECHA DE ENTREGA: No registrada
RESPUESTA ESPERADA: El pedido fue enviado y aún no registra entrega al cliente.
PEDIDO: 1b9ecfe83cdc259250e1a8aca174f0ad
ESTADO REAL: Cancelado
FECHA DE ENTREGA: No registrada
RESPUESTA ESPERADA: El pedido fue cancelado.
PEDIDO: 714fb133a6730ab81fa1d3c1b2007291
ESTADO REAL: Cancelado
FECHA DE ENTREGA: No registrada
R

## 8. Prompt para devoluciones (Ejercicio 2 - Fase 3)

In [137]:
# ============================================
# EJERCICIO 2 - CASOS DE DEVOLUCIONES
# ============================================

casos_devolucion = [
    {
        "producto": "Producto de limpieza para el hogar",
        "categoria": "Utilidades domésticas",
        "motivo": "El producto llegó en buenas condiciones, pero el cliente cambió de opinión.",
        "puede_devolver": True
    },
    {
        "producto": "Perfume",
        "categoria": "Perfumería",
        "motivo": "El cliente ya abrió y utilizó el producto.",
        "puede_devolver": False
    },
    {
        "producto": "Accesorio de computadora",
        "categoria": "Informática y accesorios",
        "motivo": "El producto llegó defectuoso.",
        "puede_devolver": True
    },
    {
        "producto": "Producto de higiene personal",
        "categoria": "Belleza y salud",
        "motivo": "El producto fue abierto después de la entrega.",
        "puede_devolver": False
    },
    {
        "producto": "Mueble para el hogar",
        "categoria": "Muebles y decoración",
        "motivo": "El producto llegó con daños visibles.",
        "puede_devolver": True
    }
]

df_devoluciones = pd.DataFrame(casos_devolucion)

display(df_devoluciones)

,producto,categoria,motivo,puede_devolver
0,Producto de limpieza para el hogar,Utilidades domésticas,"El producto llegó en buenas condiciones, pero ...",True
1,Perfume,Perfumería,El cliente ya abrió y utilizó el producto.,False
2,Accesorio de computadora,Informática y accesorios,El producto llegó defectuoso.,True
3,Producto de higiene personal,Belleza y salud,El producto fue abierto después de la entrega.,False
4,Mueble para el hogar,Muebles y decoración,El producto llegó con daños visibles.,True


In [138]:
prompt_devolucion = """
Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es ayudar a los clientes a determinar si un producto
puede ser devuelto, utilizando exclusivamente las reglas de
devolución proporcionadas.

REGLAS DE DEVOLUCIÓN:

1. Los productos que llegaron defectuosos pueden ser devueltos.
2. Los productos que llegaron dañados pueden ser devueltos.
3. Los productos de higiene personal que hayan sido abiertos
   después de la entrega no pueden ser devueltos.
4. Los perfumes que hayan sido abiertos o utilizados no pueden
   ser devueltos.
5. Un producto permitido puede ser devuelto aunque el cliente
   simplemente haya cambiado de opinión.
6. Si la información proporcionada no permite determinar si el
   producto puede devolverse, debes indicarlo claramente.
7. No inventes políticas, plazos, condiciones ni excepciones
   que no estén presentes en las reglas.
8. Responde siempre en español.
9. Utiliza un tono amable, claro y empático.
10. Explica brevemente el motivo de la decisión.

INFORMACIÓN DEL PRODUCTO:
{datos_producto}

MOTIVO DE LA DEVOLUCIÓN:
{motivo_devolucion}

RESPUESTA:

Indica claramente si el producto puede devolverse o no.
Después explica brevemente el motivo y proporciona una respuesta
empática al cliente.
"""

print(prompt_devolucion)


Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es ayudar a los clientes a determinar si un producto
puede ser devuelto, utilizando exclusivamente las reglas de
devolución proporcionadas.

REGLAS DE DEVOLUCIÓN:

1. Los productos que llegaron defectuosos pueden ser devueltos.
2. Los productos que llegaron dañados pueden ser devueltos.
3. Los productos de higiene personal que hayan sido abiertos
   después de la entrega no pueden ser devueltos.
4. Los perfumes que hayan sido abiertos o utilizados no pueden
   ser devueltos.
5. Un producto permitido puede ser devuelto aunque el cliente
   simplemente haya cambiado de opinión.
6. Si la información proporcionada no permite determinar si el
   producto puede devolverse, debes indicarlo claramente.
7. No inventes políticas, plazos, condiciones ni excepciones
   que no estén presentes en las reglas.
8. Responde siempre en español.
9. Utiliza un tono amable, claro y empático.
10. Explica brevemente el motivo de la de

## 9. Configuración del modelo de IA generativa (modelo local con Ollama)

Para este ejercicio se usa un modelo de lenguaje **open-source, gratuito y
100% local** mediante [Ollama](https://ollama.com/), en lugar de una API de
pago (Gemini u OpenAI). La rúbrica del taller permite explícitamente usar un
modelo open-source para esta fase práctica.

Pasos previos (una sola vez, fuera del notebook):

1. Instala Ollama desde `ollama.com/download` (Windows, Mac o Linux).
2. Abre una terminal y descarga un modelo, por ejemplo:
   `ollama pull llama3.2`
3. Ollama queda corriendo en segundo plano automáticamente tras instalarlo,
   escuchando en `http://localhost:11434`. Si esta sección falla con un
   error de conexión, abre la aplicación de Ollama o ejecuta `ollama serve`
   en una terminal.

Ollama expone una API compatible con la de OpenAI, así que seguimos usando
la librería `openai` como cliente, apuntándola a `http://localhost:11434/v1`
en lugar de a los servidores de OpenAI. No se necesita ninguna clave de pago.

In [139]:
# Instalar la librería openai (se usa como cliente para hablar con Ollama,
# que expone una API compatible con la de OpenAI)

!pip -q install -U openai

print("Librería openai instalada correctamente.")

Librería openai instalada correctamente.


In [140]:
# Ollama corre en tu propia máquina y no requiere una clave de pago real.
# La librería de OpenAI exige un valor no vacío, así que usamos un texto cualquiera.
api_key = "ollama"

print("Configuración de clave lista (no se necesita una clave de pago para usar Ollama).")

Configuración de clave lista (no se necesita una clave de pago para usar Ollama).


In [141]:
import openai
from openai import OpenAI

MODELO_LOCAL = "llama3.2"  # cambia este nombre si descargaste otro modelo con 'ollama pull'

cliente = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key=api_key
)

print("Conexión con el modelo local (Ollama) configurada correctamente.")

Conexión con el modelo local (Ollama) configurada correctamente.


In [142]:
# Seleccionar un pedido real de nuestra base de datos

pedido = base_prueba[
    base_prueba["numero_pedido"] == "e481f51cbdc54678b7cc49136f2d6af7"
].iloc[0]

datos_pedido = pedido.to_dict()

pregunta_cliente = "¿Cuál es el estado de mi pedido?"

print("Pedido seleccionado:")
print(datos_pedido)

Pedido seleccionado:
{'numero_pedido': 'e481f51cbdc54678b7cc49136f2d6af7', 'estado_espanol': 'Entregado', 'estado_original': 'delivered', 'fecha_compra': Timestamp('2017-10-02 10:56:33'), 'fecha_entrega_estimada': Timestamp('2017-10-18 00:00:00'), 'fecha_entrega_real': Timestamp('2017-10-10 21:25:13'), 'categoria_producto': 'Utilidades domésticas', 'precio_producto': 29.99, 'costo_transporte': 8.72, 'respuesta_esperada': 'El pedido fue entregado.'}


In [143]:
# Generar una respuesta de atención al cliente con el modelo local

prompt_final = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": prompt_final}]
    )
    print("RESPUESTA GENERADA POR EL MODELO LOCAL:")
    print(respuesta.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

RESPUESTA GENERADA POR EL MODELO LOCAL:
Señor/a,

Le informamos que se ha verificado el estado de su pedido con el número de pedido e481f51cbdc54678b7cc49136f2d6af7.

Según nuestros registros, el estado de su pedido es: **Entregado**.

Si necesita confirmar otros detalles del pedido, le agradeceremos comunicarnos con nosotros de nuevo.

Atentamente,
EcoMarket


In [144]:
# Probar la conexión con el modelo local

try:
    prueba = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": "Responde únicamente: conexión exitosa"}]
    )
    print(prueba.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

¡Excelente! La conexión exitosa es fundamental para cualquier tipo de relación, colaboración o comunicación. Es el resultado de una buena comprensión mutua, una comunicación eficaz y una actitud positiva.


In [145]:
# Generar la primera respuesta real del chatbot EcoMarket

prompt_final = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": prompt_final}]
    )
    print("RESPUESTA GENERADA POR EL MODELO LOCAL:")
    print(respuesta.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

RESPUESTA GENERADA POR EL MODELO LOCAL:
Hola, ¡le agradecemos su interés en EcoMarket!

Desafortunadamente, no podemos encontrar su pedido en la base de datos. Por favor, verifique su número de pedido (`e481f51cbdc54678b7cc49136f2d6af7`) para asegurarse de que esté correcto y vuelve a intentarlo.

Si necesita ayuda con algo más, no dude en contactarnos. Estamos aquí para ayudarle.


## 10. Pruebas de los prompts - Ejercicio 1 (los 10 pedidos)

In [146]:
resultados_chatbot = []

for _, pedido in base_prueba.iterrows():
    datos_pedido = pedido.to_dict()
    pregunta_cliente = "¿Cuál es el estado de mi pedido?"

    prompt_final = prompt_estado_pedido.format(
        datos_pedido=datos_pedido,
        pregunta_cliente=pregunta_cliente
    )

    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO_LOCAL,
            messages=[{"role": "user", "content": prompt_final}]
        )
        texto_respuesta = respuesta.choices[0].message.content
    except Exception as e:
        texto_respuesta = f"ERROR: {str(e)}"

    resultados_chatbot.append({
        "numero_pedido": pedido["numero_pedido"],
        "estado_real": pedido["estado_espanol"],
        "respuesta_modelo": texto_respuesta
    })

# Verificar cuántos resultados alcanzaron a guardarse

print("Resultados almacenados:", len(resultados_chatbot))

if len(resultados_chatbot) > 0:
    display(pd.DataFrame(resultados_chatbot))
else:
    print("No se alcanzaron a guardar resultados.")

Resultados almacenados: 10


,numero_pedido,estado_real,respuesta_modelo
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,¡Querido cliente!\n\nPodemos verificar el esta...
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,"Hola, hemos encontrado su pedido en nuestra ba..."
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,"Hola, gracias por contactarnos. Podemos confir..."
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,¡Hola! Me alegra poder ayudarte sobre el esta...
4,6942b8da583c2f9957e990d028607019,Enviado,¡Claro! Para informarte sobre el estado de tu ...
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,"Buena tarde amigo, le respondemos sobre su ped..."
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,¡Hola! Muchas gracias por contactarnos sobre e...
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,¡Hola! Gracias por contactar con EcoMarket. De...
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,**Respuesta del Asistente Virtual de Atención ...
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,¡Hola!\n\nPodemos verificar el estado de su pe...


## 11. Resultados - Ejercicio 1

In [147]:
# Guardar los resultados obtenidos hasta este momento

resultados_df = pd.DataFrame(resultados_chatbot)

archivo_resultados = RUTA_SALIDA / "Resultados_ModeloLocal_Ejercicio_1.xlsx"

with pd.ExcelWriter(archivo_resultados, engine="openpyxl") as writer:
    resultados_df.to_excel(
        writer,
        sheet_name="Resultados_ModeloLocal",
        index=False
    )

print("Resultados guardados correctamente.")
print("Pedidos procesados:", len(resultados_df))
print("Archivo:", archivo_resultados)

Resultados guardados correctamente.
Pedidos procesados: 10
Archivo: D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI\Segundo semestre\IA_Generativa\Taller_1\Salida\Resultados_ModeloLocal_Ejercicio_1.xlsx


In [148]:
# Evaluar si el modelo local respetó el estado real de los pedidos

def contiene_estado_correcto(fila):
    estado = fila["estado_real"].lower()
    respuesta = fila["respuesta_modelo"].lower()

    # Verificar que el estado real aparezca en la respuesta
    return estado in respuesta


resultados_df["estado_correcto"] = resultados_df.apply(
    contiene_estado_correcto,
    axis=1
)

print("Evaluación de los resultados:")
display(
    resultados_df[
        [
            "numero_pedido",
            "estado_real",
            "estado_correcto"
        ]
    ]
)

print("\nResumen:")
print(
    "Respuestas que respetaron el estado real:",
    resultados_df["estado_correcto"].sum(),
    "de",
    len(resultados_df)
)

Evaluación de los resultados:


,numero_pedido,estado_real,estado_correcto
0,e481f51cbdc54678b7cc49136f2d6af7,Entregado,True
1,53cdb2fc8bc7dce0b6741e2150273451,Entregado,True
2,47770eb9100c2d0c44946d9cf07ec65d,Entregado,True
3,ee64d42b8cf066f35eac1cf57de1aa85,Enviado,False
4,6942b8da583c2f9957e990d028607019,Enviado,True
5,1b9ecfe83cdc259250e1a8aca174f0ad,Cancelado,True
6,714fb133a6730ab81fa1d3c1b2007291,Cancelado,True
7,8e24261a7e58791d10cb1bf9da94df5c,No disponible,True
8,136cce7faa42fdb2cefd53fdc79a6098,Facturado,True
9,15bed8e2fec7fdbadb186b57c46c92f2,En procesamiento,False



Resumen:
Respuestas que respetaron el estado real: 8 de 10


In [149]:
# Resumen de evaluación del Ejercicio 1

total_evaluados = len(resultados_df)
correctos = resultados_df["estado_correcto"].sum()
porcentaje_correctos = (correctos / total_evaluados) * 100

print("EVALUACIÓN DEL CHATBOT - EJERCICIO 1")
print("=" * 50)
print(f"Pedidos evaluados: {total_evaluados}")
print(f"Estados respetados: {correctos}")
print(f"Porcentaje: {porcentaje_correctos:.1f}%")
print()
print("Nota: esta métrica verifica únicamente si la respuesta")
print("contiene el estado real registrado en la base de datos.")

EVALUACIÓN DEL CHATBOT - EJERCICIO 1
Pedidos evaluados: 10
Estados respetados: 8
Porcentaje: 80.0%

Nota: esta métrica verifica únicamente si la respuesta
contiene el estado real registrado en la base de datos.


## 10. Pruebas de los prompts - Ejercicio 2 (devoluciones)

In [150]:
# ============================================
# EJERCICIO 2 - PRUEBA CON EL MODELO LOCAL (OLLAMA)
# ============================================

resultados_devoluciones = []

for _, caso in df_devoluciones.iterrows():

    datos_producto = (
        f"Producto: {caso['producto']}\n"
        f"Categoría: {caso['categoria']}"
    )

    prompt_final = prompt_devolucion.format(
        datos_producto=datos_producto,
        motivo_devolucion=caso["motivo"]
    )

    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO_LOCAL,
            messages=[{"role": "user", "content": prompt_final}]
        )

        texto_respuesta = respuesta.choices[0].message.content

    except Exception as e:
        texto_respuesta = f"ERROR: {str(e)}"

    resultados_devoluciones.append({
        "producto": caso["producto"],
        "categoria": caso["categoria"],
        "motivo": caso["motivo"],
        "puede_devolver_esperado": caso["puede_devolver"],
        "respuesta_modelo": texto_respuesta
    })

    print("Caso procesado:", caso["producto"])


resultados_devoluciones_df = pd.DataFrame(resultados_devoluciones)

print("\n============================================")
print("RESULTADOS DEL EJERCICIO 2")
print("============================================")

display(
    resultados_devoluciones_df[
        [
            "producto",
            "categoria",
            "puede_devolver_esperado",
            "respuesta_modelo"
        ]
    ]
)

Caso procesado: Producto de limpieza para el hogar
Caso procesado: Perfume
Caso procesado: Accesorio de computadora
Caso procesado: Producto de higiene personal
Caso procesado: Mueble para el hogar

RESULTADOS DEL EJERCICIO 2


,producto,categoria,puede_devolver_esperado,respuesta_modelo
0,Producto de limpieza para el hogar,Utilidades domésticas,True,"Claro, puedo ayudar a resolver este asunto de ..."
1,Perfume,Perfumería,False,"Lo siento mucho, pero según nuestras reglas de..."
2,Accesorio de computadora,Informática y accesorios,True,"Lo siento mucho, pero no podemos aceptar la de..."
3,Producto de higiene personal,Belleza y salud,False,"Lo siento, pero según nuestra política de devo..."
4,Mueble para el hogar,Muebles y decoración,True,¡Hola! Me comunico con usted a través de EcoMa...


## 11. Resultados - Ejercicio 2

In [151]:
# Guardar los resultados del Ejercicio 2

archivo_resultados_devoluciones = RUTA_SALIDA / "Resultados_ModeloLocal_Ejercicio_2.xlsx"

with pd.ExcelWriter(archivo_resultados_devoluciones, engine="openpyxl") as writer:
    resultados_devoluciones_df.to_excel(
        writer,
        sheet_name="Resultados_ModeloLocal",
        index=False
    )

print("Resultados guardados correctamente.")
print("Casos procesados:", len(resultados_devoluciones_df))
print("Archivo:", archivo_resultados_devoluciones)

Resultados guardados correctamente.
Casos procesados: 5
Archivo: D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI\Segundo semestre\IA_Generativa\Taller_1\Salida\Resultados_ModeloLocal_Ejercicio_2.xlsx


## 12. Conclusiones

- El chatbot construido con un **modelo open-source local (Ollama)** logra
  responder consultas de estado de pedido y de devoluciones a partir,
  exclusivamente, de la información proporcionada en el prompt (sin
  inventar datos).
- Usar un modelo local evita costos de API y problemas de créditos/billing,
  a cambio de una calidad de respuesta que depende del tamaño del modelo
  descargado (puedes probar con modelos más grandes si tu equipo lo permite).
- Los resultados cuantitativos de los dos ejercicios quedan guardados en la
  carpeta `Salida/` (`Resultados_ModeloLocal_Ejercicio_1.xlsx` y
  `Resultados_ModeloLocal_Ejercicio_2.xlsx`) para su revisión.
- La discusión de las Fases 1 y 2 del taller (selección/justificación del
  modelo y evaluación de fortalezas, limitaciones y riesgos éticos) se
  documenta por separado en formato Markdown, tal como lo pide la rúbrica
  de entrega.